# Demo 3: Image Segmentation

**Before you start: go to Runtime → Change runtime type and select GPU.**

Object detection draws a box around each object. Segmentation goes further — 
it assigns a label to **every single pixel** in the image.

There are two main flavours:

**Semantic segmentation** labels every pixel with a category 
(sky, road, person, car...) but does not distinguish between individual instances — 
two people standing next to each other are both labelled 'person'.

**Instance segmentation** goes one step further: it distinguishes individual 
objects of the same class — person #1 gets one mask, person #2 gets another.

Segmentation is used wherever you need to understand the precise shape and 
boundary of objects, not just their location: autonomous driving (where exactly 
is the road?), medical imaging (what are the exact boundaries of this tumour?), 
satellite imagery analysis (which pixels are forest, which are farmland?), 
and image editing (select the subject, blur the background).

In this demo you will:
1. Run semantic segmentation on example images
2. Run instance segmentation and see the difference
3. Explore a medical imaging application
4. Try it on images of your own choice

## Setup

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models.segmentation import (
    deeplabv3_resnet101, DeepLabV3_ResNet101_Weights
)
from torchvision.models.detection import (
    maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
)
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import urllib.request
import colorsys

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Semantic segmentation

We use **DeepLabV3** with a ResNet-101 backbone, trained on the **Pascal VOC** 
dataset. It segments 20 categories of common objects plus background.

Unlike classification or detection, the output here is not a label or a box — 
it is a full image of the same size as the input, where each pixel value 
is a class label.

In [ ]:
# Load DeepLabV3 pretrained on Pascal VOC
seg_weights = DeepLabV3_ResNet101_Weights.COCO_WITH_VOC_LABELS_V1
seg_model   = deeplabv3_resnet101(weights=seg_weights)
seg_model   = seg_model.to(device)
seg_model.eval()

seg_preprocess = seg_weights.transforms()

# Pascal VOC class names and a colour palette
VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow', 'dining table', 'dog',
    'horse', 'motorbike', 'person', 'potted plant', 'sheep', 'sofa',
    'train', 'tv/monitor'
]

# Assign a distinct colour to each class
VOC_COLORS = np.array([
    [0,   0,   0  ],  # background — black
    [128, 0,   0  ],  # aeroplane
    [0,   128, 0  ],  # bicycle
    [128, 128, 0  ],  # bird
    [0,   0,   128],  # boat
    [128, 0,   128],  # bottle
    [0,   128, 128],  # bus
    [128, 128, 128],  # car
    [64,  0,   0  ],  # cat
    [192, 0,   0  ],  # chair
    [64,  128, 0  ],  # cow
    [192, 128, 0  ],  # dining table
    [64,  0,   128],  # dog
    [192, 0,   128],  # horse
    [64,  128, 128],  # motorbike
    [192, 128, 128],  # person
    [0,   64,  0  ],  # potted plant
    [128, 64,  0  ],  # sheep
    [0,   192, 0  ],  # sofa
    [128, 192, 0  ],  # train
    [0,   64,  128],  # tv/monitor
], dtype=np.uint8)

print(f'Semantic segmentation model loaded.')
print(f'Categories: {VOC_CLASSES}')

In [ ]:
def semantic_segment(image_path, alpha=0.6):
    """
    Run semantic segmentation and display the result.
    
    Args:
        image_path:  path to image file or URL
        alpha:       opacity of the segmentation overlay (0=transparent, 1=opaque)
    """
    # Load image
    if image_path.startswith('http'):
        urllib.request.urlretrieve(image_path, 'temp_seg.jpg')
        image_path = 'temp_seg.jpg'
    image = Image.open(image_path).convert('RGB')
    orig_size = image.size   # (width, height)
    
    # Run model
    tensor = seg_preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output     = seg_model(tensor)['out'][0]    # (num_classes, H, W)
        pred_mask  = output.argmax(dim=0).cpu().numpy()  # (H, W) class index per pixel
    
    # Resize mask back to original image size
    mask_image = Image.fromarray(pred_mask.astype(np.uint8))
    mask_image = mask_image.resize(orig_size, Image.NEAREST)
    pred_mask  = np.array(mask_image)
    
    # Convert class indices to colours
    coloured_mask = VOC_COLORS[pred_mask]   # (H, W, 3)
    
    # Blend original image with coloured mask
    orig_array = np.array(image)
    blended    = (orig_array * (1 - alpha) + coloured_mask * alpha).astype(np.uint8)
    
    # Find which classes are present
    present_classes = np.unique(pred_mask)
    
    # Display
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    axes[0].imshow(image)
    axes[0].set_title('Original image')
    axes[0].axis('off')
    
    axes[1].imshow(coloured_mask)
    axes[1].set_title('Segmentation mask')
    axes[1].axis('off')
    
    axes[2].imshow(blended)
    axes[2].set_title('Overlay')
    axes[2].axis('off')
    
    # Legend showing detected classes
    legend_patches = [
        mpatches.Patch(
            color=VOC_COLORS[c] / 255.0,
            label=VOC_CLASSES[c]
        )
        for c in present_classes if c < len(VOC_CLASSES)
    ]
    axes[2].legend(
        handles=legend_patches,
        loc='lower right', fontsize=8,
        framealpha=0.8
    )
    
    plt.suptitle('Semantic segmentation — every pixel labelled', fontsize=13)
    plt.tight_layout()
    plt.show()
    
    detected = [VOC_CLASSES[c] for c in present_classes if c < len(VOC_CLASSES)]
    print(f'Classes present: {detected}')

print('Semantic segmentation function defined.')

In [ ]:
# A street scene
semantic_segment(
    'https://upload.wikimedia.org/wikipedia/commons/thumb/0/08/Kitano_Street_Kobe01s5s4110.jpg/640px-Kitano_Street_Kobe01s5s4110.jpg'
)

In [ ]:
# A pastoral scene
semantic_segment(
    'https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/640px-Camponotus_flavomarginatus_ant.jpg'
)

## 2. Instance segmentation

Semantic segmentation treats all objects of the same class as one region. 
Instance segmentation gives each individual object its own unique mask.

We use **Mask R-CNN** — an extension of the Faster R-CNN detector from 
Demo 2 that adds a mask prediction head alongside the bounding box head. 
For each detected object, it outputs both a bounding box *and* a pixel-level mask.

This is the technique used in applications like portrait mode on smartphones 
(isolate the person, blur the background) and in medical imaging to segment 
individual cells or lesions.

In [ ]:
# Load Mask R-CNN pretrained on COCO
inst_weights = MaskRCNN_ResNet50_FPN_V2_Weights.COCO_V1
inst_model   = maskrcnn_resnet50_fpn_v2(weights=inst_weights)
inst_model   = inst_model.to(device)
inst_model.eval()

inst_preprocess = inst_weights.transforms()
COCO_CLASSES    = inst_weights.meta['categories']

print('Instance segmentation model loaded (Mask R-CNN).')

In [ ]:
def instance_segment(image_path, confidence_threshold=0.7):
    """
    Run instance segmentation and display each detected object with its own mask.
    """
    if image_path.startswith('http'):
        urllib.request.urlretrieve(image_path, 'temp_inst.jpg')
        image_path = 'temp_inst.jpg'
    image = Image.open(image_path).convert('RGB')
    
    tensor = inst_preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        predictions = inst_model(tensor)[0]
    
    # Filter by confidence
    scores = predictions['scores'].cpu().numpy()
    mask   = scores >= confidence_threshold
    boxes  = predictions['boxes'].cpu().numpy()[mask]
    labels = predictions['labels'].cpu().numpy()[mask]
    scores = scores[mask]
    masks  = predictions['masks'].cpu().numpy()[mask]  # (N, 1, H, W)
    
    # Compose a coloured instance mask overlay
    orig_array    = np.array(image)
    overlay       = orig_array.copy().astype(np.float32)
    instance_colours = []
    
    for i, (mask_i, label, score) in enumerate(zip(masks, labels, scores)):
        binary_mask = mask_i[0] > 0.5   # threshold the soft mask
        # Pick a distinct colour for this instance
        hue = (i * 0.618033988749895) % 1.0   # golden ratio for distinct hues
        rgb = np.array(colorsys.hsv_to_rgb(hue, 0.8, 0.95)) * 255
        instance_colours.append(rgb / 255.0)
        # Apply coloured mask
        for c in range(3):
            overlay[:, :, c] = np.where(
                binary_mask,
                overlay[:, :, c] * 0.4 + rgb[c] * 0.6,
                overlay[:, :, c]
            )
    
    overlay = overlay.astype(np.uint8)
    
    # Display
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
    
    ax1.imshow(image)
    ax1.set_title('Original image')
    ax1.axis('off')
    
    ax2.imshow(overlay)
    # Draw bounding boxes and labels
    for i, (box, label, score, colour) in enumerate(
            zip(boxes, labels, scores, instance_colours)):
        x1, y1, x2, y2 = box
        rect = plt.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2, edgecolor=colour, facecolor='none'
        )
        ax2.add_patch(rect)
        ax2.text(
            x1, y1 - 4,
            f'{COCO_CLASSES[label]} {score:.2f}',
            color='white', fontsize=8, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor=colour, alpha=0.8)
        )
    
    n = len(boxes)
    ax2.set_title(
        f'Instance segmentation — {n} object{"s" if n != 1 else ""} '
        f'(confidence ≥ {confidence_threshold:.0%})'
    )
    ax2.axis('off')
    
    plt.suptitle(
        'Each instance gets its own mask — objects of the same class are distinguished',
        fontsize=12
    )
    plt.tight_layout()
    plt.show()
    
    from collections import Counter
    if n > 0:
        counts = Counter(COCO_CLASSES[l] for l in labels)
        print('Detected:', ', '.join(f'{v}x {k}' for k, v in sorted(counts.items())))

print('Instance segmentation function defined.')

In [ ]:
# A group of people — instance segmentation distinguishes each individual
instance_segment(
    'https://upload.wikimedia.org/wikipedia/commons/thumb/0/08/Kitano_Street_Kobe01s5s4110.jpg/640px-Kitano_Street_Kobe01s5s4110.jpg'
)

## 3. Semantic vs instance segmentation — side by side

Let's compare the two approaches on the same image to make the 
difference concrete.

In [ ]:
url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/0/08/Kitano_Street_Kobe01s5s4110.jpg/640px-Kitano_Street_Kobe01s5s4110.jpg'

print('--- Semantic segmentation (all people = one colour) ---')
semantic_segment(url)

print('\n--- Instance segmentation (each person = different colour) ---')
instance_segment(url)

## 4. Try it on your own image

In [ ]:
# Replace with any image URL you want to try
my_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/640px-Cat03.jpg'

print('Semantic:')
semantic_segment(my_url)

print('Instance:')
instance_segment(my_url)

In [ ]:
# Or upload your own image
from google.colab import files
uploaded = files.upload()

for filename in uploaded.keys():
    print(f'\nSegmenting {filename}:')
    semantic_segment(filename)
    instance_segment(filename)

## 5. What can you do with this?

Segmentation is the right tool whenever the precise **shape** of an object 
matters, not just its location. Some project ideas where segmentation is the 
natural approach:

- **Medical imaging** — segment tumours, organs, or lesions in MRI, CT, or histology images. This is one of the most active areas of medical AI research.
- **Satellite / aerial imagery** — map land use (forest, urban, farmland, water) at pixel level from drone or satellite images
- **Agricultural monitoring** — segment individual plants or leaves to estimate crop health or yield
- **Material science** — segment grain boundaries or phases in microscopy images
- **Robotics** — segment graspable objects to plan manipulation

Note that segmentation requires **pixel-level annotations** for training, 
which are significantly more expensive to create than bounding boxes or 
class labels. For many domains, annotated segmentation datasets already exist 
— worth checking before creating your own.

---

### Think about your project

1. Is there a problem in your area of interest where you need to know the precise boundaries of objects, not just their location?
2. Does an annotated segmentation dataset already exist for your domain? (A quick search of [Papers with Code datasets](https://paperswithcode.com/datasets) is a good starting point.)
3. Would semantic segmentation be enough, or do you need to distinguish individual instances?